In [ ]:
def compute_k_w_spectrum(Mapsbox):
    dx,dy = wfs.get_dx_dy(Mapsbox[0])
    Maps_No_NaN = Mapsbox.interpolate_na(dim='y')
    Maps_dtr = wfs.detrendn(Maps_No_NaN,axes=[0,1,2])
    Maps_wdw = wfs.apply_window(Maps_dtr, Maps_dtr.dims, window_type='hanning')
    Mapshat = xfft.fft(Maps_wdw, dim=('time_counter', 'x', 'y'), dx={'x': dx, 'y': dx}, sym=True)
    Maps_psd = xfft.psd(Mapshat)
    Maps_frequency,kx,ky = wfs.get_f_kx_ky(Mapshat)
    Maps_wavenumber,kradial = wfs.get_wavnum_kradial(kx,ky)
    Maps_psd_np = Maps_psd.values
    Maps_wavenum_freq_spectrum = wfs.get_f_k_in_2D(kradial,Maps_wavenumber,Maps_psd_np)
    
    return Maps_wavenumber,Maps_frequency,Maps_wavenum_freq_spectrum

In [ ]:
def fill_nan(da):
    # INTERPOLATION OF NaNs # 
    x_axis = Axis(da.longitude.values)
    y_axis = Axis(da.latitude.values)
    t_axis = TemporalAxis(da.time.values)

    grid = Grid3D(y_axis, x_axis, t_axis, da.values.transpose(1,2,0))
    has_converged, filled = fill.gauss_seidel(grid,num_threads=n_workers)

    return da.copy(data = filled.transpose(2,0,1))

    # return  xr.DataArray(filled.transpose(2,0,1),dims=["time_counter", "x", "y"])

In [ ]:
def interp_ds(ds): 
    lon, lat = np.meshgrid(ds.longitude,ds.latitude)  

    ds_maps = ds.rename_dims({'longitude':'y','latitude':'x'})
    ds_maps = ds_maps.assign_coords({'nav_lon':(['y','x'],lon)})
    ds_maps = ds_maps.assign_coords({'nav_lat':(['y','x'],lat)})
    ds_maps = ds_maps.rename({'time':'time_counter'}) 

    return ds_maps